# Summary 18

# Домашнее задание 35
1. **Предсказание пола по имени**  
Изучите документацию API предсказания пола по имени: genderize.io/documentation. Напишите программу, которая запрашивает имя пользователя и выводит его пол и вероятность этого предсказания.

*Пример ввода:*   
Введите имя: Sasha  
Пример вывода:   
Предполагаемый пол: female  
Вероятность: 59.0%


In [4]:
import requests

name = input("Введите имя: ")

url = f"https://api.genderize.io"
params = {"name": name}

response = requests.get(url, params=params)

#data = response.json()

if response.status_code == 200:
    data = response.json()
    gender = data["gender"]
    probability = data["probability"]

    if gender:
        print(f"Предполагаемый пол: {gender}")
        print(f"Вероятность: {float(probability) * 100:.1f}%")
    else:
        print("Пол не удалось определить.")
else:
    print("Ошибка при обращении к API:", response.status_code)


Введите имя:  Nina


Предполагаемый пол: female
Вероятность: 98.0%


2. **Получение биржевых данных через API**   
Напишите программу, которая запрашивает у пользователя тикер (symbol) акций, обращается к функции с помесячными данными о результатах торгов (TIME_SERIES_MONTHLY) в API ресурса www.alphavantage.co, запрашивает результат в формате csv, записывает его в файл. Доступность ресурса предварительно нужно проверить. После этого файл требуется прочитать, извлечь данные из столбца volume и вычислить среднее значение этого столбца.

 *Пример ввода:*   
Введите тикер: BMWYY

*Пример вывода:*   
CSV-файл успешно сохранён  
Средний объём: 7638446.53


In [10]:
import requests, csv
import statistics as st

symbol = input("Введите тикер: ")

params = {
    'function' : 'TIME_SERIES_MONTHLY',
    'symbol' : symbol,
    'apikey' : 'F3Y69VXR8G9EYNI8',
    'datatype' : 'csv'
    }

url = 'https://www.alphavantage.co/query'
response = requests.get(url, params)

if response.status_code == 200:
    resp_text = response.text
else:
    print("Ошибка запроса:", response.status_code)
    exit()


with open("output_"+symbol+".csv", "w") as f:
    f.write(resp_text)  
    print("CSV-файл успешно сохранён")

volumes = []
with open("output_"+symbol+".csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader: # reader = [{'timestamp': '2026-03-26', 'open': '29.2000', 'high': '29.2000', 'low': '29.2000', 'close': '29.2000', 'volume': '0'},{'timestamp': '2026-02-27', 'open': '29.2000', 'high': '29.2000', 'low': '29.2000', 'close': '29.2000', 'volume': '0...]
        value = row.get("volume")
        volumes.append(float(value))
print(f"Средний объём: {st.mean(volumes):.2f}")


Введите тикер:  BMWYY


CSV-файл успешно сохранён
Средний объём: 7558879.38


In [9]:
volumes

[0.0,
 0.0,
 3242853.0,
 3105493.0,
 2514541.0,
 2212981.0,
 1906923.0,
 1333935.0,
 1484213.0,
 1133368.0,
 1993948.0,
 1132787.0,
 1633071.0,
 2000038.0,
 2150604.0,
 851549.0,
 879813.0,
 795875.0,
 1141167.0,
 1218186.0,
 1100067.0,
 2074132.0,
 1515595.0,
 1070262.0,
 910706.0,
 2485574.0,
 4228610.0,
 3048235.0,
 2685452.0,
 1743850.0,
 1839234.0,
 1694541.0,
 3700159.0,
 876499.0,
 1265853.0,
 1307634.0,
 1068703.0,
 1850285.0,
 949990.0,
 973693.0,
 854561.0,
 1007548.0,
 660127.0,
 1138692.0,
 1838171.0,
 1380312.0,
 947477.0,
 757463.0,
 1122794.0,
 748721.0,
 1056442.0,
 849631.0,
 1205254.0,
 1655651.0,
 2393940.0,
 3082796.0,
 5238438.0,
 1576345.0,
 1162001.0,
 1098925.0,
 854968.0,
 1239364.0,
 1180106.0,
 2359475.0,
 6085756.0,
 1856487.0,
 2044051.0,
 1091140.0,
 1262430.0,
 1420333.0,
 2063084.0,
 3333785.0,
 2277037.0,
 2635316.0,
 2207106.0,
 2099103.0,
 1478873.0,
 2477337.0,
 843013.0,
 1655667.0,
 1076951.0,
 1322514.0,
 1010030.0,
 1555201.0,
 673077.0,
 2199926

# Домашнее задание 36
1. **Поиск стран**  
Напишите программу, которая:  
Парсит таблицу с сайта https://www.iban.com/country-codes и сохраняет ее в подходящей структуре данных;  
Запрашивает у пользователя код страны (пользователь может ввести двухбуквенный или трехбуквенный код) и выводит название страны, соответствующей этому коду.

*Пример ввода:*  
Введите код страны (2 или 3 буквы):  FRA  
*Пример вывода:*  
Код FRA соответствует стране: France

In [39]:
import requests, statistics
from bs4 import BeautifulSoup

url = "https://www.iban.com/country-codes"
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    
    # На странице много таблиц. Нас интересует третья
    table = soup.find("table")

    rows = table.find_all("tr")
    
     # Получаем заголовки
    headers = [td.get_text(strip=True) for td in rows[0].find_all("th")]
#     # Получаем строки таблицы и записываем их значения в разные списки
    data = {}
    for row in rows[1:]:
        country, ab2, ab3, _ = row.find_all("td")
        data[ab2.text] = country.text
        data[ab3.text] = country.text
    code = input('Введите код страны (2 или 3 буквы):')
    print(f'Код {code} соответствует стране: {data[code]}')


else:
    print("Не удалось загрузить страницу:", response.status_code)



Введите код страны (2 или 3 буквы): FRA


Код FRA соответствует стране: France


In [40]:
data

{'AF': 'Afghanistan',
 'AFG': 'Afghanistan',
 'AX': 'Åland Islands',
 'ALA': 'Åland Islands',
 'AL': 'Albania',
 'ALB': 'Albania',
 'DZ': 'Algeria',
 'DZA': 'Algeria',
 'AS': 'American Samoa',
 'ASM': 'American Samoa',
 'AD': 'Andorra',
 'AND': 'Andorra',
 'AO': 'Angola',
 'AGO': 'Angola',
 'AI': 'Anguilla',
 'AIA': 'Anguilla',
 'AQ': 'Antarctica',
 'ATA': 'Antarctica',
 'AG': 'Antigua and Barbuda',
 'ATG': 'Antigua and Barbuda',
 'AR': 'Argentina',
 'ARG': 'Argentina',
 'AM': 'Armenia',
 'ARM': 'Armenia',
 'AW': 'Aruba',
 'ABW': 'Aruba',
 'AU': 'Australia',
 'AUS': 'Australia',
 'AT': 'Austria',
 'AUT': 'Austria',
 'AZ': 'Azerbaijan',
 'AZE': 'Azerbaijan',
 'BS': 'Bahamas (the)',
 'BHS': 'Bahamas (the)',
 'BH': 'Bahrain',
 'BHR': 'Bahrain',
 'BD': 'Bangladesh',
 'BGD': 'Bangladesh',
 'BB': 'Barbados',
 'BRB': 'Barbados',
 'BY': 'Belarus',
 'BLR': 'Belarus',
 'BE': 'Belgium',
 'BEL': 'Belgium',
 'BZ': 'Belize',
 'BLZ': 'Belize',
 'BJ': 'Benin',
 'BEN': 'Benin',
 'BM': 'Bermuda',
 'BMU'